<a href="https://colab.research.google.com/github/UERJ-FISICA/ML4PPGF_UERJ/blob/master/2026-1/Metricas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!--BOOK_INFORMATION-->
<img align="left" width= 150 style="padding-right:10px;" src="https://covers.oreillystatic.com/images/0636920052289/lrg.jpg">

*Essa aula é baseada no **capítulo 3** do livro [Hands-On Machine Learning with Scikit-Learn & TensorFlow](http://shop.oreilly.com/product/0636920052289.do) by Aurélien Geron; os notebooks do livro estão disponíveis [no GitHub](https://github.com/ageron/handson-ml2).*

# Métricas: Medidas de Performance

As métricas de performance em aprendizado de máquina são medidas quantitativas do quão bem um modelo de ML se comporta em uma tarefa específica. As métricas ajudam a estabelecer os modelos de ML mais adequados a uma tarefa, assim como podem garantir a confiabilidade de um dado modelo.

A seguir, vamos abordar os seguintes tópicos:
* Validação Cruzada (Cross-validation)
  * acurácia
* Matriz de Confusão
* Precisão e Recall
  * O score F-1
* A curva ROC

## Validação cruzada

A validação cruzada é um método de avaliação do quão bem um modelo consegue generalizar suas predições/classificações. Existem algumas técnicas de validação cruzada. Consiste em dividir as amostras de treino em treino+validação.
Entre os métodos mais usados, temos:
* **K-fold**: consiste em dividir a amostra de treino em *k* sub-conjuntos (*folds*) e treina e avalia o modelo *k* vezes, treinando (*k*-1) *folds* e avaliando no restante.
  * O resultado é um array de *k scores*
* **Stratified x-validation**: usado para garantir proporções de classes iguais
* **Shuffle and Split**:
  * Gera um número definido pelo usuário de conjuntos de treino/teste.
  * As amostras são primeiro embaralhadas e depois divididas em um par de conjuntos de treino e teste.

Mais sobre métodos de cross-validation em https://scikit-learn.org/stable/modules/cross_validation.html

### Acurácia

Vamos começar olhando para o conjunto de dados MNIST, que compreende 70.000 imagens de dígitos escritos à mão.
* Cada imagem tem o *label* do dígito correspondente


Vamos começar acessando esses dados, com o sklearn:

In [ ]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Is this notebook running on Colab or Kaggle?
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "classification"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [ ]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
mnist.keys()

Os conjuntos de dados do sklearn em geral têm um dicionário similar (vimos um parecido com os dados da flores íris!).
* Uma chave DESCR, com a descrição do conunto de dados
* Uma chave *data*, que contém um array com:
  * uma instância por linha
  * uma feature por coluna

* Uma chave *target* que contém um array com os *labels*

In [ ]:
mnist['DESCR']

In [ ]:
X, y = mnist["data"], mnist["target"]
X.shape

In [ ]:
y.shape

Temos 70.000 imagens e cada uma tem 784 features.

Isso é porque:
* cada imagem tem 28 X 28 pixels
* cada feature representa um pixel (28 pixels * 28 pixels = 784 )
  * cada pixel tem intensidade que varia de 0 (branco) a 255 (preto)

Vamos olhar para **uma** imagem do conjunto de dados:
* pegamos uma linha dos dados (um vetor de features)
* rearrumamos esse vetor em uma matriz (array) de 28 X 28
* exibimos a imagem usando a função `imshow()` do Matplotlib

In [ ]:
28 * 28

In [ ]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt

some_digit = X[0]
some_digit_image = some_digit.reshape(28, 28)
plt.imshow(some_digit_image, cmap=mpl.cm.binary)
plt.axis("off")

save_fig("some_digit_plot")
plt.show()

In [ ]:
y[0]

In [ ]:
y = y.astype(np.uint8)

In [ ]:
def plot_digit(data):
    image = data.reshape(28, 28)
    plt.imshow(image, cmap = mpl.cm.binary,
               interpolation="nearest")
    plt.axis("off")

Vamos olhar para mais alguns dados:

In [ ]:
# EXTRA
def plot_digits(instances, images_per_row=10, **options):
    size = 28
    images_per_row = min(len(instances), images_per_row)
    # This is equivalent to n_rows = ceil(len(instances) / images_per_row):
    n_rows = (len(instances) - 1) // images_per_row + 1

    # Append empty images to fill the end of the grid, if needed:
    n_empty = n_rows * images_per_row - len(instances)
    padded_instances = np.concatenate([instances, np.zeros((n_empty, size * size))], axis=0)

    # Reshape the array so it's organized as a grid containing 28×28 images:
    image_grid = padded_instances.reshape((n_rows, images_per_row, size, size))

    # Combine axes 0 and 2 (vertical image grid axis, and vertical image axis),
    # and axes 1 and 3 (horizontal axes). We first need to move the axes that we
    # want to combine next to each other, using transpose(), and only then we
    # can reshape:
    big_image = image_grid.transpose(0, 2, 1, 3).reshape(n_rows * size,
                                                         images_per_row * size)
    # Now that we have a big image, we just need to show it:
    plt.imshow(big_image, cmap = mpl.cm.binary, **options)
    plt.axis("off")

In [ ]:
plt.figure(figsize=(9,9))
example_images = X[:100]
plot_digits(example_images, images_per_row=20)
save_fig("more_digits_plot")
plt.show()

In [ ]:
y[0] # array de labels


Nosso objetivo:
* olhar para os dados de imagens de números escritos a mão
* usar um modelo classificador
* avaliar a performance do modelo, baseados apenas na acurácia
* concluir sobre a acurácia como métrica de performance de modelos de ML


Para começar, vamos criar nossos conjuntos de treino e teste para seguir adiante.


O cojnuto de dados MNIST **já está separado em conjuntos de treino** (60.000 primeiras linhas) **e teste** (10.000 últimas linhas).

In [ ]:
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

## Treinando um Classificador Binário

Como exemplo de classificador binário, vamos classificar o número 5 nas imagens que temos disponíveis.

Os vetores (arrays) de labels (targets) precisam ser criados:

In [ ]:
y_train_5 = (y_train == 5)  #True para todos targets iguais a 5, falso para o resto
y_test_5 = (y_test == 5)

In [ ]:
print(y_train_5)

Para a classificação, vamos usar a classe `SGDClassifier` -- Stochastic Gradient Descent (SGD) -- do Scikit-Learn.
* O padrão dessa classe, é um SVM linear: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.SGDClassifier.html#sklearn.linear_model.SGDClassifier

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
sgd_clf.fit(X_train, y_train_5)

In [ ]:
sgd_clf.predict([some_digit]) #some_digit = X[0]

Treinamos o classificador SGDClassifier como classificador binário de números 5 e o testamos na primeira imagem de dígito, que é um dígito 5 e o classificador recenheceu como verdadeiro.

Vamos agora avaliar a performance desse classificador.

* Começamos com a implementação de um método de validação cruzada.
  * Usaremos o método `K-fold`, implementado no Scikit-Learn
  * Usaremos a função `cross_val_score()` para avaliar nosso classificador `SGDClassifier`

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy") # 3 folds, avaliação e predição em cada fold, com modelos treinados nos folds restantes.
# O teste de performance é a acurácia.

Tivemos acima de 95% de acurácia em cada um dos folds!

* O que isso significa? Temos um modelo de excelente performance?


Vamos avaliar um outro modelo, simplório, que:
* Classifica todas as imagens como "não-5"
* Chamemos esse modelo de `Never5Classifier`

In [ ]:
from sklearn.base import BaseEstimator
class Never5Classifier(BaseEstimator):
    def fit(self, X, y=None):
        pass
    def predict(self, X):
        return np.zeros((len(X), 1), dtype=bool)

In [ ]:
never_5_clf = Never5Classifier()
cross_val_score(never_5_clf, X_train, y_train_5, cv=3, scoring="accuracy")

Todos os folds têm > 90% de acurácia!!!!

**Conclusão:**
* **A acurácia não é uma boa métrica para medir performance de um modelo!**

## Matriz de confusão

Uma ótima forma de quantificar a performance de um classificador, é se baseando na matriz de confusão:

![link text](https://github.com/UERJ-FISICA/ML4PPGF_UERJ/blob/master/pics/metrics/confusion_matrix.png?raw=1)

Outra representação da matriz de confusão:

|*Resposta*| Ruído | Sinal | ***Total*** |
|----------| ------|------ |-------------|
|*Negativo*|  <font color=red >$TN$ </font>| <font color=green> $FP$</font> |         $N$ |
|*Positivo*| <font color=green >$FN$</font> | <font color=red> $TP$</font>  |        $P$ |
|**Total** | $$B$$   |  $$S$$  |  $$Tot = N+P = S+B$$  |

* Linhas: Classes verdadeiras
  * Negativo: N = TN + FP
  * Positivo: P = FN + TP
* Colunas: Predições
  * Ruído: B = TN + FN
  * Sinal: S = FP + TP

---

### Definições :
   
   * **Precisão**: fração das instâncias de sinal corretamente classificadas como positivas. Também chamada de *pureza*.
   $$ p = \dfrac{TP}{TP+FP} = \dfrac{TP}{P}$$
   * **Eficiência (Recall / Sensitivity / True Positive Rate)**: fração do sinal que foi corretamente classificado de forma positiva.
      * Também chamado de *recall* (as vezes traducido como ___revocação___) ou **TPR** (*true positive rate*).
   $$ \epsilon = \dfrac{TP}{TP+FN} = \dfrac{TP}{S}$$


#### Outras métricas:

   * **Acurácia**: fração das instâncias classificadas corretamente (seja como sinal ou background)
$$ a = \dfrac{TP + TN}{Tot} $$
   
   * **Especificidade**: fração de background corretamente classificado de forma negativa, também conhecido de  **TNR** (*true negative rate*) e seria equivalente à eficiência em identificar o ruído (a.k.a. *background rejection*)
   $$ \epsilon_{B} = \dfrac{TN}{B} $$  
   * ***False Positive Rate***: (**FPR**) é a fração das instâncias de background classificadas incorretamente de forma positiva.
   $$ f_{FP} = \dfrac{FP}{B} = 1-\epsilon_B$$
   * ***False Negative Rate***: (**FNR**) é a fração das instâncias de sinal classificadas incorretamente de forma negativa, i.e. a *ineficiência* do sinal.
   $$ f_{FN} = \dfrac{FN}{S} = 1 - \epsilon$$

Vamos ver um exemplo de matriz de confusão, usando o `cross_val_predict()`, função do Scikit-learn que faz a predição de classificação usando o método de validação cruzada K-fold

In [ ]:
from sklearn.model_selection import cross_val_predict

y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3) #k-fold with 3 folds, used as a classifier of 5-digits (y_train_5 is a vector of True target for the digit 5)

In [ ]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_train_5, y_train_pred) # confusion matrix over the 5-digit classifier

In [ ]:
y_train_perfect_predictions = y_train_5  # pretend we reached perfection
confusion_matrix(y_train_5, y_train_perfect_predictions)

## Precisão e Recall

Olhando novamente para o nosso classificador de dígitos-5, agora com as métricas `precisão`e `recall`.

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision_score(y_train_5, y_train_pred) #precision

In [ ]:
confusion_matrix(y_train_5, y_train_pred)

In [ ]:
cm = confusion_matrix(y_train_5, y_train_pred)
cm[1, 1] / (cm[0, 1] + cm[1, 1]) # TP/(FP+TP)

In [ ]:
recall_score(y_train_5, y_train_pred) # recall

In [ ]:
cm[1, 1] / (cm[1, 0] + cm[1, 1]) # recall = TP/TP+FN

* A **precisão** indica a pureza da previsão do modelo: 83,7%
* O **recall** indica a eficiência da previsão do modelo: 65,1%

Com essas duas métricas, temos um apanhado mais real da performance do modelo.

Muitas vezes, é apropriado combinar a informação da __precisão__ e do __recall__ $\Rightarrow$ ***F1-score***

$$ \boxed {F1 = \dfrac{2}{\dfrac{1}{precisão} + \dfrac{1}{recall}} = 2 \times \dfrac{precisão \times recall}{precisão + recall} = \dfrac{TP}{TP + \dfrac{FN+FP}{2}}}$$

O Scikit-learn tem uma função que implementa o F1-score pra gente, é a função `f1_score()`:


In [ ]:
from sklearn.metrics import f1_score

f1_score(y_train_5, y_train_pred)

In [ ]:
cm[1, 1] / (cm[1, 1] + (cm[1, 0] + cm[0, 1]) / 2)

O F1-score favoriza classificadores com uma precisão e recall similares.
* Porém há casos em que precisamos de uma precisão maior
* Outros casos em que queremos uma eficiência maior


Vamos olhar para como o `SGDClassifier` faz as predições.

* O `SGDCalssifier` calcula um `score` baseado em uma `função de decisão`.
  * Cada `score` tem um valor de precisão e eficiência associado
  * Se esse `score` for maior que um dado `threshold`, a instância é classificada como pertencente à classe positiva (true)
  * Se o score for menor que o threshold, a instância é atribuída como da classe negativa
* Podemos acessar a função de decisão do `SGDClassifier`, `decision_function()`
* A `decision_function()` retorna um `score`para cada instância

In [ ]:
some_digit = X[0]
y_scores = sgd_clf.decision_function([some_digit])
print(y_scores)

Se aumentarmos o `threshold` do score no exemplo acima, o dígito X[0] (a imagem do primeiro dígito do conjunto é de um 5) será classificado como pertencente à classe negativa.

Para decidir qual threshold usar, podemos:
* Obter os scores de todas as instâncias no conjunto de treino com a função `cross_val_predict()`
* Usar a função `precision_recall_curve()` para obter os valores de precisão, recall e os thresholds para cada instância, baseados na função de decisão (score)

In [ ]:
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3,
                             method="decision_function")

In [ ]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

In [ ]:
def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):
    plt.plot(thresholds, precisions[:-1], "b--", label="Precision", linewidth=2)
    plt.plot(thresholds, recalls[:-1], "g-", label="Recall", linewidth=2)
    plt.legend(loc="center right", fontsize=16) # Not shown in the book
    plt.xlabel("Threshold", fontsize=16)        # Not shown
    plt.grid(True)                              # Not shown
    plt.axis([-50000, 50000, 0, 1])             # Not shown



recall_90_precision = recalls[np.argmax(precisions >= 0.90)]
threshold_90_precision = thresholds[np.argmax(precisions >= 0.90)]


plt.figure(figsize=(8, 4))                                                                  # Not shown
plot_precision_recall_vs_threshold(precisions, recalls, thresholds)
plt.plot([threshold_90_precision, threshold_90_precision], [0., 0.9], "r:")                 # Not shown
plt.plot([-50000, threshold_90_precision], [0.9, 0.9], "r:")                                # Not shown
plt.plot([-50000, threshold_90_precision], [recall_90_precision, recall_90_precision], "r:")# Not shown
plt.plot([threshold_90_precision], [0.9], "ro")                                             # Not shown
plt.plot([threshold_90_precision], [recall_90_precision], "ro")                             # Not shown
save_fig("precision_recall_vs_threshold_plot")                                              # Not shown
plt.show()

Outra forma de uma boa relação de precisão com recall, é olhar diretamente para a plot de **Precisão Vs. Recall**:

In [ ]:
def plot_precision_vs_recall(precisions, recalls):
    plt.plot(recalls, precisions, "b-", linewidth=2)
    plt.xlabel("Recall", fontsize=16)
    plt.ylabel("Precision", fontsize=16)
    plt.axis([0, 1, 0, 1])
    plt.grid(True)

plt.figure(figsize=(8, 6))
plot_precision_vs_recall(precisions, recalls)
plt.plot([recall_90_precision, recall_90_precision], [0., 0.9], "r:")
plt.plot([0.0, recall_90_precision], [0.9, 0.9], "r:")
plt.plot([recall_90_precision], [0.9], "ro")
save_fig("precision_vs_recall_plot")
plt.show()

In [ ]:
threshold_90_precision = thresholds[np.argmax(precisions >= 0.90)]

In [ ]:
threshold_90_precision

In [ ]:
y_train_pred_90 = (y_scores >= threshold_90_precision)

In [ ]:
precision_score(y_train_5, y_train_pred_90)

In [ ]:
recall_score(y_train_5, y_train_pred_90)

##A curva ROC

Para um modelo ser considerado como tendo uma boa performance, é necessário que haja um equilíbrio entre __precisão__ e __recall__.

A curva *Receiver Operating Characteristic* (ROC) é uma ferramenta bastante usada com lcassificadores binários.

É plotada como
* **True Positive Rate** (*Recall*) Versus **False Positive Rate**


In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)

In [ ]:
def plot_roc_curve(fpr, tpr, label=None):
    plt.plot(fpr, tpr, linewidth=2, label=label)
    plt.plot([0, 1], [0, 1], 'k--', label="random classifier") # dashed diagonal
    plt.axis([0, 1, 0, 1])                                    # Not shown in the book
    plt.xlabel('False Positive Rate (Fall-Out)', fontsize=16) # Not shown
    plt.ylabel('True Positive Rate (Recall)', fontsize=16)    # Not shown
    plt.grid(True)                                            # Not shown

plt.figure(figsize=(8, 6))                                    # Not shown
plot_roc_curve(fpr, tpr)
fpr_90 = fpr[np.argmax(tpr >= recall_90_precision)]           # Not shown
##print(recall_90_precision)
##print(fpr_90)
plt.plot([fpr_90, fpr_90], [0., recall_90_precision], "r:")   # Not shown
plt.plot([0.0, fpr_90], [recall_90_precision, recall_90_precision], "r:")  # Not shown
plt.plot([fpr_90], [recall_90_precision], "ro")               # Not shown
save_fig("roc_curve_plot")                                    # Not shown
plt.show()

* Para um classificador perfeito, a Área Sob a Curva (*Area Under the Curve*, AUC), seria igual a 1.
* Para um classificador aleatório, a AUC é igual a 0.5 (linha preta pontilhada a cima)

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_train_5, y_scores)

## Como decidir se usamos a ROC ou a curva Precisão/Recall ?

* Como regra geral:
   * Quando a classe positiva for rara, prefira a curva Precisão/Recall
      * Ou quando os **falsos positivos** forem mais importantes que os **falsos negativos**
  * Use a ROC caso contrário

## Treinando um `RandomForestClassifier`

Vamos agora treinar um outro classificador, um `RandomForestClassifier` (que veremos nas próximas aulas) para classificar os dígitos-5 e comparar com os resultados do `SGDClassifier`.

o método `predict_proba()` do `RandomForestClassifier` retorna um array contendo probabilidades de que uma dada instância (linha) pertença a uma classe (coluna).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(n_estimators=100, random_state=42)
y_probas_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3,
                                    method="predict_proba")

In [ ]:
print(y_probas_forest)

In [ ]:
y_scores_forest = y_probas_forest[:, 1] # score = proba of positive class
fpr_forest, tpr_forest, thresholds_forest = roc_curve(y_train_5,y_scores_forest)

In [ ]:
recall_for_forest = tpr_forest[np.argmax(fpr_forest >= fpr_90)]

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, "b:", linewidth=2, label="SGD")
plot_roc_curve(fpr_forest, tpr_forest, "Random Forest")
plt.plot([fpr_90, fpr_90], [0., recall_90_precision], "r:")
plt.plot([0.0, fpr_90], [recall_90_precision, recall_90_precision], "r:")
plt.plot([fpr_90], [recall_90_precision], "ro")
plt.plot([fpr_90, fpr_90], [0., recall_for_forest], "r:")
plt.plot([fpr_90], [recall_for_forest], "ro")
plt.grid(True)
plt.legend(loc="lower right", fontsize=16)
save_fig("roc_curve_comparison_plot")
plt.show()

In [ ]:
roc_auc_score(y_train_5, y_scores_forest)

In [ ]:
y_train_pred_forest = cross_val_predict(forest_clf, X_train, y_train_5, cv=3)
precision_score(y_train_5, y_train_pred_forest)

In [ ]:
recall_score(y_train_5, y_train_pred_forest)